# IIP Academy — C02 narration generator

Course: AI for Insurance Sales, Distribution & Customer Experience.
Default unit: C02_U1-1. Upload C02_U1-1_Narration_Script.md when prompted.

Derived from the existing shared IIP narration notebook on 18 October 2026. Creates slide MP3s, a full-unit MP3, estimated SRT captions, a timings CSV, a usage log and a ZIP after you run it. No audio has been generated in this file.

Run in Google Colab from top to bottom. Enable Cloud Text-to-Speech in your own Google Cloud project, configure billing, restrict an API key to that API, and store it in Colab Secrets as GOOGLE_TTS_KEY with notebook access. Configure billing alerts if appropriate; alerts do not impose a spending cap. Check current prices and your account usage before running synthesis or auditions. No free-tier or cost guarantee is made.

References: https://docs.cloud.google.com/text-to-speech/docs/chirp3-hd and https://cloud.google.com/text-to-speech/pricing

The workflow requires ffmpeg and ffprobe in the runtime. The next cell checks them. Use synthetic training content; never include customer identifiers in narration scripts. Change UNIT_CODE when moving to a different unit to avoid reusing an earlier unit's cached audio.

In [ ]:
# Cell 1 — Tools (run once per session)
!pip -q install requests
from IPython.display import Audio, display

import re, os, json, base64, subprocess, time, csv, zipfile, shutil

if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    raise RuntimeError('This workflow requires ffmpeg and ffprobe in the Colab runtime.')

def parse_script(path):
    """Return list of dicts: {num, title, text} from the narration markdown."""
    raw = open(path, encoding="utf-8").read()
    slides = []
    for m in re.finditer(r"^### SLIDE (\d+)\s*[—-]\s*(.+?)$(.*?)(?=^### SLIDE |\Z)", raw, re.M | re.S):
        num, title, body = m.group(1), m.group(2).strip(), m.group(3)
        n = re.search(r"^NARRATION:\s*$(.*?)^END NARRATION\s*$", body, re.M | re.S)
        if not n:
            raise ValueError(f"Slide {num}: NARRATION block not found")
        text = " ".join(n.group(1).split())
        slides.append({"num": int(num), "title": title, "text": text})
    if not slides:
        raise ValueError("No slides found - check the '### SLIDE 01 — Title' headings")
    return slides

def split_for_limit(text, max_bytes=4500):
    """Google TTS accepts up to 5000 bytes per request; split on sentences if needed."""
    if len(text.encode("utf-8")) <= max_bytes:
        return [text]
    parts, cur = [], ""
    for s in re.split(r"(?<=[.!?])\s+", text):
        if len((cur + " " + s).encode("utf-8")) > max_bytes and cur:
            parts.append(cur.strip()); cur = s
        else:
            cur += " " + s
    if cur.strip(): parts.append(cur.strip())
    return parts

def synth_google(text, api_key, voice, lang, rate=None):
    import requests
    body = {"input": {"text": text},
            "voice": {"languageCode": lang, "name": voice},
            "audioConfig": {"audioEncoding": "MP3"}}
    if rate: body["audioConfig"]["speakingRate"] = rate
    for attempt in range(4):
        r = requests.post("https://texttospeech.googleapis.com/v1/text:synthesize",
                          params={"key": api_key}, json=body, timeout=120)
        if r.status_code == 200:
            return base64.b64decode(r.json()["audioContent"])
        if r.status_code == 400 and rate and "rate" in r.text.lower():
            body["audioConfig"].pop("speakingRate", None); continue  # voice does not accept speakingRate
        if r.status_code in (429, 500, 503):
            time.sleep(5 * (attempt + 1)); continue
        raise RuntimeError(f"TTS error {r.status_code}: {'Request failed; check API enablement, key restrictions, billing and quota in Google Cloud.'}")
    raise RuntimeError("TTS failed after retries")

def concat_mp3(files, out, gap=0.0):
    """Concatenate mp3 files with optional silence gap, re-encoding for clean joins."""
    lst = out + ".txt"
    silence = None
    if gap > 0:
        silence = out + ".gap.mp3"
        subprocess.run(["ffmpeg","-y","-loglevel","error","-f","lavfi","-i",
                        "anullsrc=r=24000:cl=mono","-t",str(gap),"-q:a","4",silence], check=True)
    with open(lst, "w") as f:
        for i, p in enumerate(files):
            f.write(f"file '{os.path.abspath(p)}'\n")
            if silence and i < len(files) - 1:
                f.write(f"file '{os.path.abspath(silence)}'\n")
    subprocess.run(["ffmpeg","-y","-loglevel","error","-f","concat","-safe","0","-i",lst,
                    "-ar","24000","-ac","1","-b:a","96k",out], check=True)
    os.remove(lst)
    if silence: os.remove(silence)

def duration(path):
    out = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration",
                          "-of","default=nw=1:nk=1",path], capture_output=True, text=True, check=True)
    return float(out.stdout.strip())

def fmt(t):
    h, rem = divmod(t, 3600); m, s = divmod(rem, 60)
    return f"{int(h):02}:{int(m):02}:{int(s):02},{int(round((s-int(s))*1000)) % 1000:03}"

def sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def build_srt(items, out, offset_gap=0.0):
    """items: list of (text, duration). Captions per sentence, timed by character share."""
    lines, t, idx = [], 0.0, 1
    for text, dur in items:
        sents = sentences(text); total = sum(len(s) for s in sents)
        st = t
        for s in sents:
            d = dur * len(s) / total
            lines.append(f"{idx}\n{fmt(st)} --> {fmt(st+d)}\n{s}\n"); idx += 1; st += d
        t += dur + offset_gap
    open(out, "w", encoding="utf-8").write("\n".join(lines))

In [ ]:
# Cell 2 — Settings
from google.colab import userdata
API_KEY = userdata.get("GOOGLE_TTS_KEY")
if not API_KEY:
    raise RuntimeError('Add GOOGLE_TTS_KEY to Colab Secrets and enable notebook access.')

UNIT_CODE   = "C02_U1-1"                     # used in file names
VOICE       = "en-GB-Chirp3-HD-Charon"       # change after the audition in Cell 4
LANGUAGE    = "en-GB"                        # must match the first part of the voice name
SPEAKING_RATE = None                         # e.g. 0.95 for slightly slower; None = voice default
GAP_SECONDS = 0.8                            # pause between slides in the full-unit MP3
REDO_SLIDES = []                             # e.g. [4, 11] to regenerate only those slides
OUT_DIR     = f"/content/{UNIT_CODE}_audio"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Cell 3 — Upload the narration script (.md) and preview cost
from google.colab import files
up = files.upload()
SCRIPT_PATH = list(up.keys())[0]
slides = parse_script(SCRIPT_PATH)
chars = sum(len(s["text"]) for s in slides); words = sum(len(s["text"].split()) for s in slides)
print(f"{len(slides)} slides | {words:,} words | {chars:,} characters | about {words/150:.1f} minutes of audio")
print("Review current Google Cloud pricing and your account usage before synthesis. No spending cap is enforced here.")
for s in slides:
    print(f"  Slide {s['num']:02} – {s['title']} ({len(s['text'].split())} words)")

In [ ]:
# Cell 4 — Voice audition (uses only slide 1; about 600 characters per voice)
import requests
v = requests.get("https://texttospeech.googleapis.com/v1/voices", params={"key": API_KEY}, timeout=60).json()
chirp = sorted(x["name"] for x in v.get("voices", []) if "Chirp3-HD" in x["name"] and x["name"].startswith(("en-GB","en-US")))
print(f"{len(chirp)} English Chirp 3 HD voices available, e.g.:", ", ".join(chirp[:12]), "...")

CANDIDATES = ["en-GB-Chirp3-HD-Charon", "en-GB-Chirp3-HD-Kore", "en-US-Chirp3-HD-Aoede", "en-US-Chirp3-HD-Orus"]
sample = slides[0]["text"]
for name in CANDIDATES:
    if name not in chirp:
        print("Not available, skipping:", name); continue
    audio = synth_google(sample, API_KEY, name, name[:5], SPEAKING_RATE)
    path = f"{OUT_DIR}/audition_{name}.mp3"; open(path, "wb").write(audio)
    print(name); display(Audio(path))
with open(f"{OUT_DIR}/usage_log.csv", "a", newline="") as f:
    csv.writer(f).writerow([time.strftime("%Y-%m-%d %H:%M"), UNIT_CODE, "audition", len(sample) * len(CANDIDATES)])

In [ ]:
# Cell 5 — Generate one MP3 per slide (skips slides already done unless listed in REDO_SLIDES)
used = 0
for s in slides:
    path = f"{OUT_DIR}/{UNIT_CODE}_slide_{s['num']:02}.mp3"
    if os.path.exists(path) and s["num"] not in REDO_SLIDES:
        continue
    parts = split_for_limit(s["text"])
    tmp = []
    for i, p in enumerate(parts):
        t = f"{path}.part{i}.mp3"; open(t, "wb").write(synth_google(p, API_KEY, VOICE, LANGUAGE, SPEAKING_RATE)); tmp.append(t)
        used += len(p)
    if len(tmp) == 1: os.replace(tmp[0], path)
    else:
        concat_mp3(tmp, path); [os.remove(t) for t in tmp]
    print(f"Slide {s['num']:02} done ({duration(path):.1f}s)")

with open(f"{OUT_DIR}/usage_log.csv", "a", newline="") as f:
    csv.writer(f).writerow([time.strftime("%Y-%m-%d %H:%M"), UNIT_CODE, VOICE, used])
print(f"Characters used this run: {used:,}")

In [ ]:
# Cell 6 — Listen to any slide before assembling (change the number)
display(Audio(f"{OUT_DIR}/{UNIT_CODE}_slide_{1:02}.mp3"))

In [ ]:
# Cell 7 — Build full-unit MP3, captions, PowerPoint timings, and download everything as a zip
slide_files = [f"{OUT_DIR}/{UNIT_CODE}_slide_{s['num']:02}.mp3" for s in slides]
durs = [duration(p) for p in slide_files]
full = f"{OUT_DIR}/{UNIT_CODE}_full.mp3"
concat_mp3(slide_files, full, gap=GAP_SECONDS)
build_srt([(s["text"], d) for s, d in zip(slides, durs)], f"{OUT_DIR}/{UNIT_CODE}_full.srt", offset_gap=GAP_SECONDS)

with open(f"{OUT_DIR}/{UNIT_CODE}_slide_timings.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["slide", "title", "audio_seconds", "advance_after_seconds"])
    for s, d in zip(slides, durs):
        w.writerow([s["num"], s["title"], round(d, 2), round(d + GAP_SECONDS, 1)])

print(f"Full unit: {duration(full)/60:.1f} minutes")
zpath = f"/content/{UNIT_CODE}_audio.zip"
with zipfile.ZipFile(zpath, "w") as z:
    for fn in os.listdir(OUT_DIR):
        if not fn.startswith("audition_"): z.write(f"{OUT_DIR}/{fn}", fn)
files.download(zpath)

## Putting the audio into the slides

**Option A — PowerPoint (simplest):** on each slide, *Insert → Audio → Audio on My PC* and choose that slide's MP3; under *Playback* set **Start: Automatically** and tick **Hide During Show**. Under *Transitions*, untick *On Mouse Click* and set **After** to the `advance_after_seconds` value from the timings CSV. Then *File → Export → Create a Video* (Full HD, use recorded timings). Upload the `.srt` file to the learning platform as captions.

**Option B — send the slides (PPTX or PDF) and the zip back to Claude** to have them stitched into an MP4 with captions.

## Fixing pronunciation
Chirp 3: HD voices do not take SSML tags, so fix problems in the script itself: spell abbreviations as letters with hyphens (I-R-A, D-P-P-A), write numbers as words where needed ("two hundred million Uganda shillings"), and respell difficult names phonetically. Then add the slide number to `REDO_SLIDES` and rerun Cells 5 and 7.